# 01 — C4.5: categorías, números y valores faltantes

Implementación propia de las fórmulas y la construcción del árbol: pandas
organiza los datos, sin delegar el aprendizaje a bibliotecas de ML.
La base categórica y numérica sigue SEM04-DecisionTrees, filminas 23–30, y
Liu (2011), pp. 70–76. HU-08 incorpora pesos fraccionarios e inferencia
probabilística para atributos faltantes (guía del proyecto, sección 8).

**Contrato:** `fit(X, y)` recibe DataFrames; `y` tiene una columna `objetivo`
sin nulos. `predict(X)` devuelve una columna `prediccion`, conservando índice
y orden. `predict_proba(X)` devuelve una columna por clase en el orden de
`classes_`, definido por primera aparición en entrenamiento.

Se admiten `None`, `NaN`, `pd.NA` y `NaT` en atributos. No hay imputación.
La prepoda combina `threshold` (bits), `max_depth`, `min_samples_split` (masa)
y `min_gain_ratio` (adimensional); no se implementa pospoda. Las decisiones están
en `docs/hu08_decisiones.md` y `docs/hu07_decisiones.md`.

Ejecutar todas las celdas desde la raíz del repositorio o desde `notebooks/`.
En Colab se necesita el repositorio completo y sus dependencias.
`datos_tp.py` comparte carga y partición con RF; allí scikit-learn solo carga
Iris y separa train/test.


In [ ]:
from dataclasses import dataclass, field
from math import fsum, isfinite, log2
from numbers import Integral, Real

import pandas as pd
from pandas.api.types import is_bool_dtype, is_complex_dtype, is_numeric_dtype, is_scalar


# Un dato faltante puede llegar como None, NaN, pd.NA o NaT según la columna.
# Se centraliza acá para que todo el árbol lo reconozca de la misma manera.
def es_faltante(valor):
    """Reconoce None, NaN, pd.NA y NaT sin confundir categorías compuestas."""
    return is_scalar(valor) and bool(pd.isna(valor))


## 1. Impureza, ganancia y pesos fraccionarios

Para una clase $c$, $p_c$ es su masa dividida por la masa total del nodo.
Con pesos unitarios se recuperan los cálculos de HU-02/HU-03:

$$H(D)=-\sum_c p_c\log_2 p_c$$
$$Gain(D,A)=H(D)-\sum_j (W_j/W)H(D_j)$$
$$SplitInfo(D,A)=-\sum_j (W_j/W)\log_2(W_j/W)$$

Si el atributo tiene faltantes, se calcula la ganancia solo sobre los conocidos
$D_K$ y se ajusta por su proporción de masa $W_K/W$:

$$Gain^*(D,A)=\frac{W_K}{W}\left[H(D_K)-\sum_j\frac{W_j}{W_K}H(D_j)\right]$$
$$SplitInfo^*(D,A)=-\sum_j\frac{W_j}{W}\log_2\frac{W_j}{W}
-\frac{W_U}{W}\log_2\frac{W_U}{W}$$
$$GainRatio^*(D,A)=Gain^*(D,A)/SplitInfo^*(D,A)$$

$W_U$ es la masa desconocida: contribuye al denominador como grupo adicional,
pero no se crea una rama para nulos. Cada caso faltante de peso $w$ se reparte
entre los hijos con pesos $w W_j/W_K$. Se conserva la masa por clase.
Se usa $0\log 0=0$; no se eligen particiones con menos de dos ramas conocidas.


In [ ]:
# Cada caso tiene un peso: 1 si está completo y una fracción si un faltante lo
# repartió entre ramas. Sin pesos, un caso repartido contaría varias veces.
def preparar_pesos(etiquetas, pesos=None):
    """Materializa etiquetas y pesos finitos no negativos, alineados por posición."""
    etiquetas = list(etiquetas)
    pesos = [1.0] * len(etiquetas) if pesos is None else list(pesos)
    if len(etiquetas) != len(pesos):
        raise ValueError("Cada etiqueta debe tener su peso correspondiente.")
    if any(not isinstance(p, Real) or isinstance(p, bool) or not isfinite(p) or p < 0
           for p in pesos):
        raise ValueError("Los pesos deben ser reales finitos y no negativos.")
    return etiquetas, pesos


# Suma cuánta masa tiene cada clase en un nodo. Es la base de las probabilidades.
# fsum evita que se acumulen errores de redondeo al sumar fracciones.
def conteos_ponderados(etiquetas, pesos):
    """Suma masa por clase, sin contar las copias fraccionarias como casos enteros."""
    grupos = {}
    for clase, peso in zip(etiquetas, pesos):
        if peso > 0:
            grupos.setdefault(clase, []).append(peso)
    return {clase: fsum(grupo) for clase, grupo in grupos.items()}


# Mide qué tan mezcladas están las clases: 0 si son todas iguales y máxima si
# están equilibradas. Un buen corte es el que deja hijos con poca entropía.
def entropia(etiquetas, pesos=None):
    """H(D) = -sum(p_c log2(p_c)); p_c usa masa, no cantidad de filas."""
    etiquetas, pesos = preparar_pesos(etiquetas, pesos)
    conteos = conteos_ponderados(etiquetas, pesos)
    total = fsum(conteos.values())
    if total == 0:
        return 0.0
    return -fsum((n / total) * log2(n / total) for n in conteos.values())


# Entropía promedio de los hijos, pesada por su tamaño. Sin ese peso, un hijo
# con un solo caso importaría igual que uno con cien.
def entropia_ponderada(grupos, pesos_grupos=None):
    """Entropía posterior: sum(W_j / W * H(D_j)), con pesos opcionales."""
    grupos = [list(grupo) for grupo in grupos]
    if pesos_grupos is None:
        pesos_grupos = [[1.0] * len(grupo) for grupo in grupos]
    else:
        pesos_grupos = list(pesos_grupos)
    if len(grupos) != len(pesos_grupos):
        raise ValueError("Cada grupo debe tener sus pesos correspondientes.")
    preparados = [preparar_pesos(g, p) for g, p in zip(grupos, pesos_grupos)]
    total = fsum(fsum(p) for _, p in preparados)
    if total == 0:
        return 0.0
    return fsum(fsum(p) / total * entropia(g, p) for g, p in preparados)


# Cuánta incertidumbre elimina un corte: entropía del padre menos la de los hijos.
def ganancia_informacion(etiquetas, grupos, pesos=None, pesos_grupos=None):
    """H(D) - H_A(D). Precondición: los grupos particionan D y sus pesos."""
    return entropia(etiquetas, pesos) - entropia_ponderada(grupos, pesos_grupos)


# Misma fórmula de entropía, pero sobre tamaños de ramas. Se usa para SplitInfo.
def entropia_masas(masas):
    """Entropía de masas no negativas; los términos de masa cero no contribuyen."""
    masas = list(masas)
    total = fsum(masas)
    if total == 0:
        return 0.0
    return -fsum((m / total) * log2(m / total) for m in masas if m > 0)


# Mide en cuántas partes y qué tan parejo divide un atributo. Sirve para castigar
# atributos con muchos valores, que dan ganancia alta sin aportar generalización.
def split_info(grupos, pesos_grupos=None):
    """Entropía de tamaños/masas de las ramas, sin valores desconocidos."""
    grupos = [list(g) for g in grupos]
    if pesos_grupos is None:
        return entropia_masas([len(g) for g in grupos])
    pesos_grupos = list(pesos_grupos)
    if len(grupos) != len(pesos_grupos):
        raise ValueError("Cada grupo debe tener sus pesos correspondientes.")
    return entropia_masas([fsum(preparar_pesos(g, p)[1])
                          for g, p in zip(grupos, pesos_grupos)])


# Criterio de C4.5: ganancia dividida por SplitInfo. Se usa en lugar de la ganancia
# sola para no favorecer a los atributos con muchos valores.
def gain_ratio(etiquetas, grupos, pesos=None, pesos_grupos=None):
    """Gain / SplitInfo sin faltantes; denominador cero devuelve 0 defensivo."""
    denominador = split_info(grupos, pesos_grupos)
    if denominador == 0:
        return 0.0
    return ganancia_informacion(etiquetas, grupos, pesos, pesos_grupos) / denominador


# Calcula todas las métricas de un corte candidato, teniendo en cuenta los faltantes.
# Los casos sin valor no pueden evaluarse, por eso la ganancia se reduce según la
# proporción de casos conocidos: un atributo incompleto merece menos confianza.
def evaluar_particion(etiquetas, indices_por_rama, pesos=None, faltantes=()):
    """C4.5 con faltantes: Gain = (W_conocido/W) * Gain_conocido.

    Las ramas contienen SOLO conocidos. SplitInfo incluye la masa desconocida
    como un grupo adicional (no como una rama del árbol). Las posiciones son
    locales al nodo; no se usan los índices pandas ni los datos de test.
    """
    etiquetas, pesos = preparar_pesos(etiquetas, pesos)
    grupos = [[etiquetas[i] for i in pos] for pos in indices_por_rama.values()]
    pesos_grupos = [[pesos[i] for i in pos] for pos in indices_por_rama.values()]
    conocidos = [i for pos in indices_por_rama.values() for i in pos]
    masa_conocida = fsum(pesos[i] for i in conocidos)
    masa_faltante = fsum(pesos[i] for i in faltantes)
    total = masa_conocida + masa_faltante
    posterior = entropia_ponderada(grupos, pesos_grupos)
    ganancia = 0.0
    if total > 0 and masa_conocida > 0:
        ganancia = masa_conocida / total * ganancia_informacion(
            [etiquetas[i] for i in conocidos], grupos,
            [pesos[i] for i in conocidos], pesos_grupos,
        )
    denominador = entropia_masas([fsum(p) for p in pesos_grupos] + [masa_faltante])
    return {
        "entropia_ponderada": posterior,
        "ganancia": ganancia,
        "split_info": denominador,
        "gain_ratio": ganancia / denominador if denominador > 0 else 0.0,
        "ramas": indices_por_rama,
        "faltantes": list(faltantes),
        "proporciones": {rama: fsum(pesos[i] for i in pos) / masa_conocida
                         for rama, pos in indices_por_rama.items()} if masa_conocida else {},
    }


## 2. Categorías y cortes numéricos con faltantes

Las categorías conocidas generan una rama por valor observado. Los números
conocidos se ordenan y cada valor distinto salvo el mayor propone un corte
`A <= v_i` / `A > v_i`, aprendido solo con los casos del nodo de entrenamiento.
Los faltantes no crean valores candidatos ni categorías artificiales.

Se compara el Gain Ratio ajustado de las particiones candidatas. En un empate
exacto se conserva el primer atributo por orden de columnas y el menor corte.
Los atributos totalmente faltantes o constantes entre conocidos se descartan.


In [ ]:
# Corte para atributos de categorías: una rama por cada valor observado en el nodo.
# Devuelve None con menos de dos ramas, porque ese corte no separaría nada.
def particion_categorica(valores, etiquetas, pesos=None):
    """Una rama por categoría conocida con masa positiva; nulos por separado."""
    valores = list(valores)
    etiquetas, pesos = preparar_pesos(etiquetas, pesos)
    if len(valores) != len(etiquetas):
        raise ValueError("Cada valor debe tener su etiqueta correspondiente.")
    ramas, faltantes = {}, []
    for posicion, (valor, peso) in enumerate(zip(valores, pesos)):
        if peso == 0:
            continue
        # Si falta el dato de la pregunta, no se puede elegir una rama: se combinan las
        # respuestas de todas, pesadas según cuántos casos pasaron por cada una.
        if es_faltante(valor):
            faltantes.append(posicion)
        else:
            ramas.setdefault(valor, []).append(posicion)
    if len(ramas) < 2:
        return None
    return evaluar_particion(etiquetas, ramas, pesos, faltantes)


# Corte para atributos numéricos: prueba cada valor distinto como umbral y se queda
# con el de mayor razón de ganancia. El mayor valor se omite porque dejaría una rama vacía.
def buscar_mejor_umbral(valores, etiquetas, pesos=None, por_ganancia=False):
    """Evalúa A <= v_i y A > v_i sobre valores conocidos distintos del nodo.

    Se maximiza Gain Ratio ajustado por faltantes o, con por_ganancia, la ganancia.
    Los empates exactos conservan el menor corte; los nulos no proponen cortes,
    pero penalizan la ganancia.
    """
    valores = list(valores)
    etiquetas, pesos = preparar_pesos(etiquetas, pesos)
    if len(valores) != len(etiquetas):
        raise ValueError("Cada valor debe tener su etiqueta correspondiente.")
    conocidos, faltantes = [], []
    for i, (valor, peso) in enumerate(zip(valores, pesos)):
        if peso == 0:
            continue
        if es_faltante(valor):
            faltantes.append(i)
        elif not isinstance(valor, Real) or isinstance(valor, bool) or not isfinite(valor):
            raise ValueError("Los valores numéricos conocidos deben ser reales y finitos.")
        else:
            conocidos.append(i)
    candidatos = sorted({valores[i] for i in conocidos})[:-1]
    # C4.5 elige el umbral por ganancia. Con razón de ganancia ganarían los cortes que
    # separan un solo caso, porque su SplitInfo es muy chico aunque aporten poco.
    criterio = "ganancia" if por_ganancia else "gain_ratio"
    mejor = None
    for umbral in candidatos:
        ramas = {
            "<=": [i for i in conocidos if valores[i] <= umbral],
            ">": [i for i in conocidos if valores[i] > umbral],
        }
        evaluacion = evaluar_particion(etiquetas, ramas, pesos, faltantes)
        evaluacion["umbral"] = umbral
        if mejor is None or evaluacion[criterio] > mejor[criterio]:
            mejor = evaluacion
    return mejor


## 3. Nodo y conservación de masa

Un nodo guarda su clase mayoritaria, conteos ponderados, entropía y masa efectiva
`peso_total`. `n_ejemplos` cuenta las filas representadas, que pueden incluir
copias fraccionarias. Los hijos conservan, sumados, la masa de cada clase del padre.

Las decisiones guardan atributo, corte, métricas y `proporciones` de las ramas.
Los empates de clase siguen el orden global `classes_`. Una categoría nueva
devuelve la distribución local del nodo alcanzado, cuyo argmax es su mayoría.


In [ ]:
@dataclass
# Un nodo del árbol. Se guarda lo que se necesita para predecir y también las métricas
# del corte, para poder explicar y dibujar por qué el árbol decidió así.
class Nodo:
    """Decisión u hoja con conteos fraccionarios y probabilidades de sus ramas.

    n_ejemplos cuenta filas representadas; peso_total es la masa efectiva.
    Las copias de un caso faltante suman su peso original entre los hijos.
    """
    clase: object
    conteos: dict
    n_ejemplos: int
    entropia: float
    peso_total: float = 0.0
    atributo: object = None
    tipo: str | None = None
    umbral: float | int | None = None
    hijos: dict = field(default_factory=dict)
    proporciones: dict = field(default_factory=dict)
    ganancia: float | None = None
    split_info: float | None = None
    gain_ratio: float | None = None
    motivo_parada: str | None = None

    @property
    def es_hoja(self):
        return not self.hijos


## 4. Construcción y predicción probabilística

`fit` inicia cada fila con peso 1. La selección usa métricas ponderadas y cada
faltante se incorpora a todos los hijos con el peso proporcional de su rama.
Los atributos categóricos usados se retiran; los numéricos pueden reutilizarse.

Se crea una hoja ante clase pura, atributos agotados, profundidad máxima,
masa efectiva menor que `min_samples_split`, ninguna partición válida,
ganancia ajustada menor que `threshold` o Gain Ratio menor que `min_gain_ratio`.
`threshold`, la masa y el ratio paran con `<` (igualdad permite dividir);
`max_depth` se detiene al alcanzar el límite. `threshold` está en bits y
`min_gain_ratio` es adimensional; ambos deben pasar. Con muchos nulos conviene
prepoda positiva para limitar la expansión de copias.

En predicción, un valor conocido sigue una rama. Un faltante mezcla las
distribuciones completas de todas las ramas según sus proporciones aprendidas.
Una categoría nueva usa las frecuencias locales del nodo. No se votan las clases
mayoritarias de los hijos. `predict` es el argmax de `predict_proba`; los empates
eligen la primera clase observada globalmente en `fit`.

Las columnas numéricas, excepto booleanos, se tratan como números; texto,
booleanos y `category` son categóricos. Los códigos categóricos de Heart ya
se convierten a texto en la preparación compartida. Los nulos del objetivo,
infinitos y valores complejos se rechazan.


In [ ]:
# El árbol de decisión propio. Sigue la interfaz fit/predict de scikit-learn para poder
# usarlo igual que Random Forest en las comparaciones.
class ArbolC45:
    """C4.5 didáctico con prepoda estructural y faltantes ponderados.

    Los pesos fraccionarios son internos; fit inicia cada fila con peso 1.
    threshold se aplica a la ganancia AJUSTADA, en bits, con comparación <.
    max_depth usa raíz=0 y se detiene en >=; min_samples_split compara masa
    efectiva, no filas; min_gain_ratio es adimensional. Masa y ratio paran
    con < (igualdad permite dividir). No hay pospoda; empates: classes_.
    """

    # Los cuatro parámetros limitan el crecimiento (prepoda) para reducir el sobreajuste.
    # Por defecto no limitan nada: el árbol crece hasta separar todos los casos posibles.
    def __init__(self, threshold=0.0, max_depth=None, min_samples_split=2,
                 min_gain_ratio=0.0, filtro_ganancia=True):
        self.threshold = threshold
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_gain_ratio = min_gain_ratio
        # Activa las dos reglas de Quinlan contra los cortes desparejos: umbral por
        # ganancia y competencia solo entre atributos con ganancia al menos promedio.
        self.filtro_ganancia = filtro_ganancia
        self.raiz_ = None
        self._validar_parametros()

    def _validar_threshold(self):
        if (not isinstance(self.threshold, Real) or isinstance(self.threshold, bool)
                or not isfinite(self.threshold) or self.threshold < 0):
            raise ValueError("threshold debe ser un número real finito y no negativo.")

    # Se valida al crear y al entrenar para que un valor inválido falle con un mensaje claro.
    def _validar_parametros(self):
        """Valida en construcción y de nuevo en fit los cuatro controles.

        min_samples_split es masa efectiva, no cantidad de filas: las copias
        fraccionarias de HU-08 no deben inflar la evidencia de un nodo.
        """
        self._validar_threshold()
        if (self.max_depth is not None
                and (not isinstance(self.max_depth, Integral)
                     or isinstance(self.max_depth, bool) or self.max_depth < 0)):
            raise ValueError("max_depth debe ser None o un entero no negativo.")
        if (not isinstance(self.min_samples_split, Real)
                or isinstance(self.min_samples_split, bool)
                or not isfinite(self.min_samples_split) or self.min_samples_split < 2):
            raise ValueError("min_samples_split debe ser real finito y mayor o igual que 2.")
        if (not isinstance(self.min_gain_ratio, Real)
                or isinstance(self.min_gain_ratio, bool)
                or not isfinite(self.min_gain_ratio) or self.min_gain_ratio < 0):
            raise ValueError("min_gain_ratio debe ser real finito y no negativo.")
        if not isinstance(self.filtro_ganancia, bool):
            raise ValueError("filtro_ganancia debe ser True o False.")

    @staticmethod
    # Revisa la entrada antes de entrenar o predecir, porque un error en este punto sería
    # mucho más difícil de encontrar en medio de la recursión.
    def _validar_X(X):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("X debe ser un pandas.DataFrame.")
        if X.empty:
            raise ValueError("X debe contener casos y atributos.")
        if not X.columns.is_unique or X.columns.isna().any():
            raise ValueError("Los nombres de los atributos deben ser únicos y no nulos.")
        for columna in X.columns:
            serie = X[columna]
            if is_complex_dtype(serie.dtype):
                raise ValueError("No se admiten atributos de números complejos.")
            for valor in serie:
                if es_faltante(valor):
                    continue
                if isinstance(valor, complex) or (isinstance(valor, Real) and not isfinite(valor)):
                    raise ValueError("Los números conocidos deben ser reales y finitos.")
                try:
                    hash(valor)
                except TypeError as error:
                    raise ValueError("Las categorías deben ser valores simples/hashables.") from error

    # Entrena el árbol. Primero valida y decide qué columnas son numéricas o categóricas,
    # porque cada tipo se corta de manera distinta. Cada caso arranca con peso 1.
    def fit(self, X, y):
        """Entrena con X (admite faltantes) e y DataFrame objetivo sin faltantes."""
        self._validar_parametros()
        self._validar_X(X)
        if not isinstance(y, pd.DataFrame):
            raise TypeError("y debe ser un pandas.DataFrame.")
        if list(y.columns) != ["objetivo"] or not X.index.equals(y.index):
            raise ValueError("y debe tener la columna objetivo y el mismo índice y orden que X.")
        if y.isna().any().any():
            raise ValueError("y no debe contener valores faltantes.")
        etiquetas = y["objetivo"].tolist()
        try:
            clases = list(dict.fromkeys(etiquetas))
        except TypeError as error:
            raise ValueError("Las clases deben ser valores simples/hashables.") from error

        self.classes_ = pd.Index(clases, tupleize_cols=False)
        self.columnas_ = X.columns.copy()
        self.tipos_ = {
            columna: ("numerico" if is_numeric_dtype(X[columna].dtype)
                      and not is_bool_dtype(X[columna].dtype) else "categorico")
            for columna in X.columns
        }
        self.raiz_ = self._construir(X, etiquetas, list(X.columns), [1.0] * len(X))
        return self

    # Corazón del algoritmo: crea un nodo y, si corresponde dividirlo, se llama a sí mismo
    # para cada rama. Termina cuando se cumple alguna condición de parada.
    def _construir(self, X, etiquetas, atributos, pesos, profundidad=0):
        masa = conteos_ponderados(etiquetas, pesos)
        # Orden global estable para empates, también en nodos descendientes.
        conteos = {c: masa[c] for c in self.classes_ if c in masa}
        clase = max(conteos, key=conteos.get)
        nodo = Nodo(clase, conteos, len(etiquetas), entropia(etiquetas, pesos),
                    peso_total=fsum(conteos.values()))
        # Parada: todos los casos son de la misma clase, no hay nada más que separar.
        if len(conteos) == 1:
            nodo.motivo_parada = "clase_pura"
            return nodo
        if not atributos:
            nodo.motivo_parada = "sin_atributos"
            return nodo
        # Paradas baratas antes de evaluar todos los cortes candidatos.
        if self.max_depth is not None and profundidad >= self.max_depth:
            nodo.motivo_parada = "profundidad_maxima"
            return nodo
        # Masa efectiva: las copias de faltantes no son evidencia entera nueva.
        if fsum(pesos) < self.min_samples_split:
            nodo.motivo_parada = "masa_menor_que_min_samples_split"
            return nodo

        # Evalúa todos los atributos disponibles y guarda el mejor corte de cada uno.
        candidatas = []
        for atributo in atributos:
            valores = X[atributo].tolist()
            if self.tipos_[atributo] == "numerico":
                candidata = buscar_mejor_umbral(valores, etiquetas, pesos, self.filtro_ganancia)
            else:
                candidata = particion_categorica(valores, etiquetas, pesos)
            if candidata is None or candidata["split_info"] == 0:
                continue
            candidatas.append(dict(candidata, atributo=atributo))
        # Quinlan: la razón de ganancia favorece cortes con SplitInfo muy chico aunque aporten
        # poco. Por eso solo compiten los atributos con ganancia al menos promedio.
        if candidatas and self.filtro_ganancia:
            promedio = fsum(c["ganancia"] for c in candidatas) / len(candidatas)
            candidatas = [c for c in candidatas if c["ganancia"] >= promedio - 1e-12]
        # Entre los que quedan gana la mayor razón de ganancia; en un empate, el primero.
        mejor = None
        for candidata in candidatas:
            if mejor is None or candidata["gain_ratio"] > mejor["gain_ratio"]:
                mejor = candidata
        if mejor is None:
            nodo.motivo_parada = "sin_particion_valida"
            return nodo
        # Prepoda: si el mejor corte aporta poco, se deja el nodo como hoja.
        if mejor["ganancia"] < self.threshold:
            nodo.motivo_parada = "ganancia_menor_que_threshold"
            return nodo
        # Chequeo independiente y adimensional, con la misma regla de igualdad.
        if mejor["gain_ratio"] < self.min_gain_ratio:
            nodo.motivo_parada = "gain_ratio_menor_que_min_gain_ratio"
            return nodo

        atributo = mejor["atributo"]
        nodo.atributo = atributo
        nodo.tipo = self.tipos_[atributo]
        nodo.umbral = mejor.get("umbral")
        nodo.ganancia = mejor["ganancia"]
        nodo.split_info = mejor["split_info"]
        nodo.gain_ratio = mejor["gain_ratio"]
        nodo.proporciones = mejor["proporciones"]
        # Un atributo categórico ya no sirve después de usarse, porque en cada rama vale lo mismo.
        # Uno numérico sí puede volver a usarse con otro umbral.
        siguientes = ([a for a in atributos if a != atributo]
                      if nodo.tipo == "categorico" else atributos)
        # Crea un hijo por rama. Un caso con el dato faltante entra en todas las ramas con un
        # peso proporcional a su tamaño: así no se descarta y la masa total se conserva.
        for rama, conocidos in mejor["ramas"].items():
            # Cada faltante participa en todos los hijos con peso w * q_rama.
            # Se conserva el orden original; se descartan solo pesos cero.
            conocidos = set(conocidos)
            faltantes = set(mejor["faltantes"])
            posiciones, pesos_hijo = [], []
            for i, peso in enumerate(pesos):
                nuevo = peso if i in conocidos else (
                    peso * nodo.proporciones[rama] if i in faltantes else 0.0
                )
                if nuevo > 0:
                    posiciones.append(i)
                    pesos_hijo.append(nuevo)
            nodo.hijos[rama] = self._construir(
                X.iloc[posiciones], [etiquetas[i] for i in posiciones], siguientes,
                pesos_hijo, profundidad + 1,
            )
        return nodo

    # Recorre el árbol para un caso y devuelve la probabilidad de cada clase.
    def _distribucion_fila(self, fila, nodo):
        if nodo.es_hoja:
            return [nodo.conteos.get(c, 0.0) / nodo.peso_total for c in self.classes_]
        valor = fila[nodo.atributo]
        if es_faltante(valor):
            # Mezcla de distribuciones completas, no voto de clases mayoritarias.
            hijas = [(nodo.proporciones[r], self._distribucion_fila(fila, hijo))
                     for r, hijo in nodo.hijos.items()]
            return [fsum(q * dist[j] for q, dist in hijas) for j in range(len(self.classes_))]
        rama = ("<=" if valor <= nodo.umbral else ">") if nodo.tipo == "numerico" else valor
        # Categoría que no apareció en entrenamiento: se responde con lo que se sabe en este nodo.
        if rama not in nodo.hijos:
            # Categoría nueva: distribución LOCAL; argmax devuelve su mayoría.
            return [nodo.conteos.get(c, 0.0) / nodo.peso_total for c in self.classes_]
        return self._distribucion_fila(fila, nodo.hijos[rama])

    def _validar_prediccion(self, X):
        if self.raiz_ is None:
            raise RuntimeError("El árbol debe entrenarse antes de predecir.")
        self._validar_X(X)
        if not X.columns.equals(self.columnas_):
            raise ValueError("X debe tener las mismas columnas y orden que en entrenamiento.")
        for atributo, tipo in self.tipos_.items():
            if tipo == "numerico" and any(
                not es_faltante(v) and (not isinstance(v, Real) or isinstance(v, bool))
                for v in X[atributo]
            ):
                raise ValueError(f"El atributo {atributo!r} debe conservar su tipo numérico.")

    # Devuelve las probabilidades por clase para cada caso, sin modificar X.
    def predict_proba(self, X: pd.DataFrame) -> pd.DataFrame:
        """Probabilidades por classes_; conserva índice/orden y no modifica X.

        Son frecuencias ponderadas del árbol, no probabilidades calibradas.
        Una entrada completamente faltante combina todas las rutas alcanzables.
        """
        self._validar_prediccion(X)
        distribuciones = [self._distribucion_fila(dict(zip(self.columnas_, valores)), self.raiz_)
                          for valores in X.itertuples(index=False, name=None)]
        # Normalización final para absorber únicamente el redondeo acumulado.
        distribuciones = [[p / fsum(fila) for p in fila] for fila in distribuciones]
        return pd.DataFrame(distribuciones, columns=self.classes_.copy(), index=X.index.copy())

    # Devuelve la clase más probable. Es la función común con Random Forest.
    def predict(self, X: pd.DataFrame) -> pd.DataFrame:
        """Argmax de predict_proba; empates: primera clase observada en fit."""
        probabilidades = self.predict_proba(X)
        clases = [self.classes_[max(range(len(fila)), key=fila.__getitem__)]
                  for fila in probabilidades.itertuples(index=False, name=None)]
        return pd.DataFrame({"prediccion": clases}, index=X.index.copy())


## 5. Comprobación con el ejemplo de clase

Se transcribe la tabla de **15 casos de la filmina 27**; `ID` es índice, no
atributo. Para `Own_house` hay 6 casos `Yes` en la rama verdadera y 6 `No` +
3 `Yes` en la falsa (filmina 24). El cálculo manual es:

- H(D) = −(6/15) log₂(6/15) − (9/15) log₂(9/15) ≈ **0,970951**.
- H_falsa = −(6/9) log₂(6/9) − (3/9) log₂(3/9) ≈ **0,918296**.
- H_ponderada = (6/15) × 0 + (9/15) × 0,918296 ≈ **0,550978**.
- Gain ≈ **0,419973**; Split Info ≈ **0,970951**; Gain Ratio ≈ **0,432538**.

Las aserciones usan valores de referencia, no una segunda llamada a la función
como resultado esperado. Se verifica además un corte numérico perfecto.

In [ ]:
# Comprueba las fórmulas contra un ejemplo de clase calculado a mano.
# Para qué: si la entropía o la ganancia estuvieran mal, todo el árbol estaría mal.
from io import StringIO
from math import isclose
from IPython.display import display

ejemplo_clase = pd.read_csv(StringIO("""ID Age Has_job Own_house Credit_rating objetivo
1 young false false fair No
2 young false false good No
3 young true false good Yes
4 young true true fair Yes
5 young false false fair No
6 middle false false fair No
7 middle false false good No
8 middle true true good Yes
9 middle false true excellent Yes
10 middle false true excellent Yes
11 old false true excellent Yes
12 old false true good Yes
13 old true false good Yes
14 old true false excellent Yes
15 old false false fair No
"""), sep=r"\s+", dtype=str).set_index("ID")
X_clase = ejemplo_clase.drop(columns="objetivo")
y_clase = ejemplo_clase[["objetivo"]]
evaluaciones = []
for atributo in X_clase:
    evaluacion = particion_categorica(X_clase[atributo].tolist(), y_clase["objetivo"].tolist())
    evaluaciones.append({"atributo": atributo, **{k: v for k, v in evaluacion.items() if k != "ramas"}})
tabla_ganancias = pd.DataFrame(evaluaciones).set_index("atributo")
display(tabla_ganancias)

assert isclose(entropia(y_clase["objetivo"]), 0.9709505944546686, abs_tol=1e-12)
for metrica, esperado in {
    "entropia_ponderada": 0.5509775004326937,
    "ganancia": 0.4199730940219749,
    "split_info": 0.9709505944546686,
    "gain_ratio": 0.4325380677663126,
}.items():
    assert isclose(tabla_ganancias.loc["Own_house", metrica], esperado, abs_tol=1e-12)
arbol_clase = ArbolC45().fit(X_clase, y_clase)
assert arbol_clase.raiz_.atributo == "Own_house"
assert arbol_clase.predict(X_clase)["prediccion"].tolist() == y_clase["objetivo"].tolist()

corte = buscar_mejor_umbral([1, 2, 3, 4], ["No", "No", "Yes", "Yes"])
assert corte["umbral"] == 2
assert corte["ganancia"] == corte["split_info"] == corte["gain_ratio"] == 1.0
print("OK: cálculos manuales, ejemplo de clase y corte numérico.")

## 6. Datasets y particiones compartidas con Random Forest

Se reutiliza exactamente la carga de Contact Lenses, Iris y Heart Disease.
La partición sigue siendo 80/20 estratificada, con semilla 42. Los dos modelos
reciben los mismos casos originales; el one-hot sigue siendo interno de RF.

In [ ]:
# Carga los datasets con las mismas funciones que Random Forest.
# Por qué: la comparación solo es justa si ambos modelos usan los mismos casos.
from pathlib import Path
import sys

# Compartir la carga y las mismas particiones sin ejecutar el otro notebook.
for carpeta in (Path.cwd(), *Path.cwd().parents):
    if (carpeta / "datos_tp.py").is_file() and (carpeta / "data/raw").is_dir():
        RAIZ = carpeta
        break
else:
    raise FileNotFoundError("Ejecutá desde la raíz del repositorio o notebooks/.")
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from datos_tp import (
    SEMILLA, FRACCION_TEST, TAMANOS, CLASES, cargar_datasets, particionar,
)

datasets = cargar_datasets()
particiones = {nombre: particionar(datos["X"], datos["y"])
               for nombre, datos in datasets.items()}

## 7. Ejecución base en los tres datasets

`THRESHOLD = 0.0` permite modificar la prepoda desde código; el deslizador será
parte de otra HU. Cambiar el parámetro requiere volver a entrenar.

Se muestra accuracy de entrenamiento y prueba como comprobación inicial,
sin exigir un mínimo ni elegir el parámetro mirando test. Para explorar
parametrizaciones después se necesitará validación dentro de entrenamiento.
Contact Lenses tiene solo cinco casos de prueba: cada acierto cambia 20 puntos
porcentuales la accuracy. Estos resultados no prueban superioridad sobre RF.

In [ ]:
# Entrena el árbol en los tres datasets y muestra una primera accuracy.
THRESHOLD = 0.0
modelos = {}
predictores = {}
predicciones = {}
resultados = []
for nombre, parte in particiones.items():
    modelo = ArbolC45(threshold=THRESHOLD).fit(parte["X_train"], parte["y_train"])
    salida_train = modelo.predict(parte["X_train"])
    salida_test = modelo.predict(parte["X_test"])
    modelos[nombre] = modelo
    predictores[nombre] = modelo.predict
    predicciones[nombre] = salida_test
    resultados.append({
        "dataset": nombre, "train": len(parte["X_train"]), "test": len(parte["X_test"]),
        "accuracy_train": (salida_train["prediccion"] == parte["y_train"]["objetivo"]).mean(),
        "accuracy_test": (salida_test["prediccion"] == parte["y_test"]["objetivo"]).mean(),
    })
    print(f"{nombre}: primeras predicciones")
    display(parte["y_test"].join(salida_test).head())
resultados = pd.DataFrame(resultados).set_index("dataset")
display(resultados)

## 8. Contrato y casos límite

Se comprueba el orden de las predicciones y la ausencia de modificaciones en X.
El caso XOR verifica que una ganancia inicial cero pueda conducir a una buena
división en el siguiente nivel, cuando `threshold=0`. El ejemplo numérico
comprueba que un atributo pueda repetirse en un camino y que `≤` incluya el corte.

La suite `tests/test_c45.py` amplía estas comprobaciones con entradas inválidas,
empates, atributos constantes, agotamiento de categorías, datasets mixtos y
reentrenamiento. Carga las mismas celdas de definiciones: no mantiene otra copia
del algoritmo.

In [ ]:
# Comprueba que la entrada y la salida respeten el contrato y los casos límite.
for nombre, parte in particiones.items():
    entrada = parte["X_test"].iloc[::-1].copy(deep=True)
    original = entrada.copy(deep=True)
    salida = predictores[nombre](entrada)
    assert list(salida.columns) == ["prediccion"]
    assert salida.index.equals(entrada.index)
    assert set(salida["prediccion"]) <= set(parte["y_train"]["objetivo"])
    pd.testing.assert_frame_equal(entrada, original)
    pd.testing.assert_frame_equal(salida, predicciones[nombre].iloc[::-1])
    repetida = particionar(datasets[nombre]["X"], datasets[nombre]["y"])
    for clave in parte:
        pd.testing.assert_frame_equal(parte[clave], repetida[clave])

X_xor = pd.DataFrame({"a": ["0", "0", "1", "1"], "b": ["0", "1", "0", "1"]})
y_xor = pd.DataFrame({"objetivo": ["No", "Yes", "Yes", "No"]})
sin_prepoda = ArbolC45(threshold=0).fit(X_xor, y_xor)
assert sin_prepoda.raiz_.ganancia == 0
assert sin_prepoda.predict(X_xor)["prediccion"].tolist() == y_xor["objetivo"].tolist()
assert ArbolC45(threshold=0.01).fit(X_xor, y_xor).raiz_.es_hoja

X_intervalo = pd.DataFrame({"x": [1, 2, 3, 4, 5, 6]})
y_intervalo = pd.DataFrame({"objetivo": ["A", "A", "B", "B", "A", "A"]})
intervalo = ArbolC45().fit(X_intervalo, y_intervalo)
assert intervalo.predict(X_intervalo)["prediccion"].tolist() == y_intervalo["objetivo"].tolist()
assert intervalo.raiz_.atributo == "x"
assert any(hijo.atributo == "x" for hijo in intervalo.raiz_.hijos.values())
print("OK: contrato en tres datasets, prepoda y reutilización de atributos numéricos.")

## 9. HU-08: ejemplo manual y pruebas sobre datasets con nulos

El ejemplo pequeño verifica a mano que `x=[1,2,None]`, `y=[A,B,A]` produce
`P(A|x faltante)=2/3`, `P(B)=1/3`. Después se inyecta un 5% de nulos en las
características de entrenamiento y test de los tres datasets con semilla fija.
La máscara no usa etiquetas ni modifica las particiones originales.

La prepoda `threshold=0.05` es una configuración de demostración, no un parámetro
elegido sobre test. Se verifica normalización, clases e índices; las métricas
comparativas y porcentajes sistemáticos corresponden a HU-12.


In [ ]:
# Comprueba el manejo de faltantes: un ejemplo manual y datos con nulos agregados.
import numpy as np

X_manual = pd.DataFrame({"x": [1, 2, None]})
y_manual = pd.DataFrame({"objetivo": ["A", "B", "A"]})
modelo_manual = ArbolC45().fit(X_manual, y_manual)
p_manual = modelo_manual.predict_proba(pd.DataFrame({"x": [None]}))
assert isclose(p_manual.iloc[0]["A"], 2 / 3)
assert isclose(p_manual.iloc[0]["B"], 1 / 3)
display(p_manual)

generador_nulos = np.random.default_rng(42)
for nombre, parte in particiones.items():
    X_train = parte["X_train"].copy(deep=True)
    X_test = parte["X_test"].copy(deep=True)
    for entrada in (X_train, X_test):
        mascara = generador_nulos.random(entrada.shape) < 0.05
        entrada[entrada.columns] = entrada.mask(mascara)
    modelo_nulos = ArbolC45(threshold=0.05).fit(X_train, parte["y_train"])
    # Incluir además una consulta con todos los atributos faltantes.
    consulta = X_test.copy(deep=True)
    consulta.iloc[0] = None
    probabilidades = modelo_nulos.predict_proba(consulta)
    etiquetas = modelo_nulos.predict(consulta)
    assert probabilidades.index.equals(consulta.index)
    assert probabilidades.columns.equals(modelo_nulos.classes_)
    assert np.allclose(probabilidades.sum(axis=1), 1)
    assert ((probabilidades >= 0) & (probabilidades <= 1)).all().all()
    assert etiquetas["prediccion"].tolist() == probabilidades.idxmax(axis=1).tolist()
    pd.testing.assert_frame_equal(datasets[nombre]["X"].loc[parte["X_train"].index], parte["X_train"])
    print(f"OK HU-08 — {nombre}: entrenamiento con nulos e inferencia ponderada.")


## 10. HU-07: max_depth, min_samples_split y min_gain_ratio

Los tres controles se aplican durante el crecimiento; no se implementa pospoda.
`max_depth=None` no limita, la raíz está en profundidad 0 y un nodo con
`profundidad >= max_depth` queda como hoja con sus conteos ponderados y la
política de clase existente. `min_samples_split=2` compara la **masa efectiva**
`sum(weights)`, no la cantidad de filas: las copias fraccionarias de HU-08 no
inflan la evidencia. `min_gain_ratio=0` compara el Gain Ratio adimensional,
separado de `threshold`, que sigue midiendo ganancia de información en bits.
La igualdad permite dividir con masa y ratio; `max_depth` se detiene al
límite. Ambos umbrales deben pasar.
Son decisiones del proyecto, no requisitos de una biblioteca. Ver
`docs/hu07_decisiones.md`.


In [ ]:
# Comprueba que cada parámetro de prepoda realmente cambie el árbol.
from math import isclose

def medir_arbol(nodo, profundidad=0):
    """Devuelve (profundidad máxima, cantidad de hojas) recursivamente."""
    if nodo.es_hoja:
        return profundidad, 1
    medidas = [medir_arbol(hijo, profundidad + 1) for hijo in nodo.hijos.values()]
    return max(p for p, _ in medidas), sum(h for _, h in medidas)


X_control = pd.DataFrame({"g": ["a", "a", "b", "b"], "h": ["u", "v", "u", "v"]})
y_control = pd.DataFrame({"objetivo": ["A", "B", "B", "A"]})
configuraciones = [
    (None, 2, 0.0),
    (0, 2, 0.0),
    (1, 2, 0.0),
    (None, 2.01, 0.0),
    (None, 2, 0.6),
]
filas = []
for max_depth, min_samples_split, min_gain_ratio in configuraciones:
    modelo = ArbolC45(max_depth=max_depth, min_samples_split=min_samples_split,
                      min_gain_ratio=min_gain_ratio).fit(X_control, y_control)
    profundidad, hojas = medir_arbol(modelo.raiz_)
    filas.append({"max_depth": max_depth, "min_samples_split": min_samples_split,
                  "min_gain_ratio": min_gain_ratio, "profundidad": profundidad,
                  "hojas": hojas, "motivo_raiz": modelo.raiz_.motivo_parada})
tabla_controles = pd.DataFrame(filas)
display(tabla_controles)
assert tabla_controles[["profundidad", "hojas"]].values.tolist() == [
    [2, 4], [0, 1], [1, 2], [1, 2], [0, 1],
]

# La copia faltante suma una fila, pero no una unidad de masa: el hijo b
# tiene 2 filas y masa 1.5, así que el mínimo por defecto 2 lo deja hoja.
X_masa = pd.DataFrame({"g": ["a", "b", None], "h": ["u", "u", "v"]})
y_masa = pd.DataFrame({"objetivo": ["A", "B", "A"]})
arbol_masa = ArbolC45().fit(X_masa, y_masa)
hijo_b = arbol_masa.raiz_.hijos["b"]
assert hijo_b.n_ejemplos == 2 and isclose(hijo_b.peso_total, 1.5)
assert hijo_b.es_hoja and hijo_b.motivo_parada == "masa_menor_que_min_samples_split"
assert isclose(arbol_masa.predict_proba(pd.DataFrame({"g": ["b"], "h": ["v"]})).iloc[0]["B"], 2 / 3)
print("OK HU-07: profundidad, masa efectiva y Gain Ratio modifican el árbol.")


## Fuentes y límites de esta entrega

- SEM04-DecisionTrees, filminas 23–30: árbol base, fórmulas y cortes numéricos.
- Liu (2011), pp. 70–76: recursión, métricas y continuos.
- Guía del proyecto, sección 8 (Liu pp. 78–79): manejo probabilístico de faltantes.
- `docs/hu08_decisiones.md`: fórmulas, recomendaciones, contrato y casos especiales.
- `docs/hu07_decisiones.md`: controles de crecimiento, masa efectiva y compatibilidad.
- `README.md` y `datos_tp.py`: fuentes y preparación compartida de los datasets.

Las probabilidades son frecuencias ponderadas, sin calibración. El cambio de
desempates locales a globales puede modificar respuestas de hojas empatadas.
La ejecución local no acredita por sí sola ejecución en Colab. Pospoda,
interfaz y comparación experimental sistemática quedan para otras HUs.
